# Population Stability Index (PSI) Calculation and Visualization

## Learning Objectives

In this notebook, you will learn:
- How to calculate the Population Stability Index (PSI) from scratch
- How to interpret PSI values
- How to visualize distribution changes using PSI
- The impact of binning strategies on PSI values

## Introduction

The Population Stability Index (PSI) is a widely used metric in the financial industry for monitoring feature drift. It quantifies the change in the distribution of a variable between two datasets, typically a reference dataset (e.g., training data) and a monitored dataset (e.g., recent production data).

PSI is calculated by binning the variable and comparing the percentage of observations in each bin between the two datasets. The formula is:

$$PSI = \sum_{i=1}^{n} (Actual_i - Expected_i) \times \ln\left(\frac{Actual_i}{Expected_i}\right)$$

Where:
- $Actual_i$ is the percentage of observations in bin $i$ for the monitored dataset
- $Expected_i$ is the percentage of observations in bin $i$ for the reference dataset
- $n$ is the number of bins

### Interpretation Thresholds

- **PSI < 0.1**: No significant shift
- **0.1 ≤ PSI < 0.2**: Moderate shift
- **PSI ≥ 0.2**: Significant shift

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Generate Synthetic Data

We'll create two datasets to simulate a reference (training) dataset and a monitored (production) dataset with some drift.

In [ ]:
# Generate reference dataset (training data)
reference_data = np.random.normal(loc=50, scale=10, size=10000)

# Generate monitored dataset with drift (production data)
# Scenario 1: Small shift in mean
monitored_data_small_shift = np.random.normal(loc=52, scale=10, size=10000)

# Scenario 2: Moderate shift in mean
monitored_data_moderate_shift = np.random.normal(loc=55, scale=10, size=10000)

# Scenario 3: Large shift in mean and variance
monitored_data_large_shift = np.random.normal(loc=60, scale=15, size=10000)

print("Reference data statistics:")
print(f"Mean: {reference_data.mean():.2f}, Std: {reference_data.std():.2f}")
print("\nMonitored data (small shift) statistics:")
print(f"Mean: {monitored_data_small_shift.mean():.2f}, Std: {monitored_data_small_shift.std():.2f}")
print("\nMonitored data (moderate shift) statistics:")
print(f"Mean: {monitored_data_moderate_shift.mean():.2f}, Std: {monitored_data_moderate_shift.std():.2f}")
print("\nMonitored data (large shift) statistics:")
print(f"Mean: {monitored_data_large_shift.mean():.2f}, Std: {monitored_data_large_shift.std():.2f}")

## 2. Implement PSI Calculation

We'll implement the PSI calculation from scratch to understand the mechanics of the metric.

In [ ]:
def calculate_psi(reference, monitored, bins=10, binning_strategy='quantile'):
    """
    Calculate Population Stability Index (PSI) between two distributions.
    
    Parameters:
    -----------
    reference : array-like
        Reference dataset (e.g., training data)
    monitored : array-like
        Monitored dataset (e.g., production data)
    bins : int or array-like
        Number of bins or bin edges
    binning_strategy : str
        'quantile' for equal-frequency bins or 'uniform' for equal-width bins
    
    Returns:
    --------
    psi_value : float
        PSI value
    bin_df : DataFrame
        DataFrame with bin-wise statistics
    """
    # Determine bin edges based on reference data
    if binning_strategy == 'quantile':
        # Equal-frequency binning
        bin_edges = np.percentile(reference, np.linspace(0, 100, bins + 1))
    else:
        # Equal-width binning
        bin_edges = np.linspace(reference.min(), reference.max(), bins + 1)
    
    # Ensure unique bin edges
    bin_edges = np.unique(bin_edges)
    
    # Calculate bin counts for reference and monitored data
    reference_counts, _ = np.histogram(reference, bins=bin_edges)
    monitored_counts, _ = np.histogram(monitored, bins=bin_edges)
    
    # Calculate percentages
    reference_percents = reference_counts / len(reference)
    monitored_percents = monitored_counts / len(monitored)
    
    # Add small epsilon to avoid division by zero and log(0)
    epsilon = 1e-10
    reference_percents = np.where(reference_percents == 0, epsilon, reference_percents)
    monitored_percents = np.where(monitored_percents == 0, epsilon, monitored_percents)
    
    # Calculate PSI
    psi_values = (monitored_percents - reference_percents) * np.log(monitored_percents / reference_percents)
    psi_value = np.sum(psi_values)
    
    # Create DataFrame with bin-wise statistics
    bin_df = pd.DataFrame({
        'Bin': range(len(bin_edges) - 1),
        'Lower_Bound': bin_edges[:-1],
        'Upper_Bound': bin_edges[1:],
        'Reference_Count': reference_counts,
        'Monitored_Count': monitored_counts,
        'Reference_Percent': reference_percents * 100,
        'Monitored_Percent': monitored_percents * 100,
        'PSI_Contribution': psi_values
    })
    
    return psi_value, bin_df

## 3. Calculate PSI for Different Drift Scenarios

In [ ]:
# Calculate PSI for small shift
psi_small, bin_df_small = calculate_psi(reference_data, monitored_data_small_shift, bins=10)
print(f"PSI (Small Shift): {psi_small:.4f}")
print("Interpretation: ", end="")
if psi_small < 0.1:
    print("No significant shift")
elif psi_small < 0.2:
    print("Moderate shift")
else:
    print("Significant shift")

print("\n" + "="*50 + "\n")

# Calculate PSI for moderate shift
psi_moderate, bin_df_moderate = calculate_psi(reference_data, monitored_data_moderate_shift, bins=10)
print(f"PSI (Moderate Shift): {psi_moderate:.4f}")
print("Interpretation: ", end="")
if psi_moderate < 0.1:
    print("No significant shift")
elif psi_moderate < 0.2:
    print("Moderate shift")
else:
    print("Significant shift")

print("\n" + "="*50 + "\n")

# Calculate PSI for large shift
psi_large, bin_df_large = calculate_psi(reference_data, monitored_data_large_shift, bins=10)
print(f"PSI (Large Shift): {psi_large:.4f}")
print("Interpretation: ", end="")
if psi_large < 0.1:
    print("No significant shift")
elif psi_large < 0.2:
    print("Moderate shift")
else:
    print("Significant shift")

## 4. Visualize Distribution Changes

In [ ]:
# Create visualization function
def visualize_psi(reference, monitored, psi_value, title):
    """
    Visualize the distribution comparison and PSI.
    """
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot 1: Distribution comparison
    axes[0].hist(reference, bins=30, alpha=0.5, label='Reference', density=True, color='blue')
    axes[0].hist(monitored, bins=30, alpha=0.5, label='Monitored', density=True, color='red')
    axes[0].set_xlabel('Value')
    axes[0].set_ylabel('Density')
    axes[0].set_title(f'{title}\nPSI = {psi_value:.4f}')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Bin-wise PSI contribution
    _, bin_df = calculate_psi(reference, monitored, bins=10)
    axes[1].bar(bin_df['Bin'], bin_df['PSI_Contribution'], color='green', alpha=0.7)
    axes[1].set_xlabel('Bin')
    axes[1].set_ylabel('PSI Contribution')
    axes[1].set_title('PSI Contribution by Bin')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Visualize all scenarios
visualize_psi(reference_data, monitored_data_small_shift, psi_small, 'Small Shift')
visualize_psi(reference_data, monitored_data_moderate_shift, psi_moderate, 'Moderate Shift')
visualize_psi(reference_data, monitored_data_large_shift, psi_large, 'Large Shift')

## 5. Impact of Binning Strategy

The PSI value can be sensitive to the choice of binning strategy. Let's explore how different binning strategies affect the PSI calculation.

In [ ]:
# Test different number of bins
bin_counts = [5, 10, 15, 20, 25, 30]
psi_quantile = []
psi_uniform = []

for n_bins in bin_counts:
    psi_q, _ = calculate_psi(reference_data, monitored_data_moderate_shift, bins=n_bins, binning_strategy='quantile')
    psi_u, _ = calculate_psi(reference_data, monitored_data_moderate_shift, bins=n_bins, binning_strategy='uniform')
    psi_quantile.append(psi_q)
    psi_uniform.append(psi_u)

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(bin_counts, psi_quantile, marker='o', label='Quantile Binning', linewidth=2)
plt.plot(bin_counts, psi_uniform, marker='s', label='Uniform Binning', linewidth=2)
plt.axhline(y=0.1, color='orange', linestyle='--', label='PSI = 0.1 (Moderate Threshold)')
plt.axhline(y=0.2, color='red', linestyle='--', label='PSI = 0.2 (Significant Threshold)')
plt.xlabel('Number of Bins')
plt.ylabel('PSI Value')
plt.title('Impact of Binning Strategy on PSI')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("PSI values for different binning strategies:")
print("\nQuantile Binning:")
for n_bins, psi in zip(bin_counts, psi_quantile):
    print(f"  {n_bins} bins: {psi:.4f}")
print("\nUniform Binning:")
for n_bins, psi in zip(bin_counts, psi_uniform):
    print(f"  {n_bins} bins: {psi:.4f}")

## 6. Practical Example: Credit Scoring

Let's apply PSI to a practical example in credit scoring, where we monitor the distribution of applicant income over time.

In [ ]:
# Simulate credit applicant income data
# Reference period: Pre-recession
income_reference = np.random.lognormal(mean=10.5, sigma=0.5, size=5000)

# Monitored period: Post-recession (lower average income)
income_monitored = np.random.lognormal(mean=10.3, sigma=0.6, size=5000)

# Calculate PSI
psi_income, bin_df_income = calculate_psi(income_reference, income_monitored, bins=10)

print(f"PSI for Income Distribution: {psi_income:.4f}")
print(f"\nInterpretation: {'Significant shift - Model retraining recommended' if psi_income >= 0.2 else 'Moderate shift - Monitor closely' if psi_income >= 0.1 else 'No significant shift'}")

# Display bin-wise statistics
print("\nBin-wise Statistics:")
print(bin_df_income.to_string(index=False))

# Visualize
visualize_psi(income_reference, income_monitored, psi_income, 'Credit Applicant Income Distribution')

## Key Takeaways

1. **PSI is a simple and interpretable metric** for detecting distribution shifts in a single variable.

2. **Interpretation thresholds** provide a rule of thumb for assessing the severity of drift:
   - PSI < 0.1: No significant shift
   - 0.1 ≤ PSI < 0.2: Moderate shift
   - PSI ≥ 0.2: Significant shift

3. **Binning strategy matters**: The choice of binning strategy and number of bins can affect the PSI value. Quantile binning is often preferred as it ensures each bin has a similar number of observations in the reference dataset.

4. **PSI is univariate**: It can only detect drift in one feature at a time. For multivariate drift detection, other methods should be used.

5. **Business context is crucial**: PSI thresholds should be calibrated based on the specific business context and the cost of model failures.

## Exercises

1. Modify the code to calculate PSI for a categorical variable (hint: use frequency counts instead of binning).

2. Implement a function to automatically select the optimal number of bins based on Doane's formula.

3. Create a time-series plot showing how PSI evolves over multiple time periods.

4. Compare PSI with other drift detection metrics (e.g., KS test) on the same datasets.